# 02 · Evaluación: WER, RTF, coste y comparativas
Transcribe todos los audios, calcula WER (donde haya referencia en `REF_DIR/<nombre>.txt`), RTF y coste,
y lo registra en MLflow (experimento `asr_poc`).
Para comparar modelos, cambia `BACKEND`/`MODEL_ID` en `utils/00_config`, repite y mira **Experiments → Compare**.

In [0]:
%pip install -q cactus-compute transformers accelerate soundfile jiwer pandas mlflow imageio-ffmpeg
dbutils.library.restartPython()

In [0]:
%run ./utils/00_config

## A. Evaluación sobre los audios originales
Una fila por audio (duración, tiempo de proceso, RTF, WER) y un run en MLflow.

In [0]:
asr = load_asr()
df_opus, m_opus = run_eval(asr, audio_files(), f"{BACKEND}_opus")
display(df_opus[["file", "audio_s", "proc_s", "rtf", "wer"]])
print(f"RTF global: {m_opus['rtf']:.3f} (<1 = más rápido que tiempo real)")
if "wer_corpus" not in m_opus:
    print("Sin transcripciones de referencia: añade .txt en", REF_DIR)

## B. WAV frente a OPUS
Se transcodifica el mismo audio a WAV 16 kHz mono. **Ojo**: con Whistle el audio ya se convierte a WAV
internamente, así que la diferencia de WER será ~0 y la de tiempo es el coste de ffmpeg.
Con el baseline Whisper la comparación sí es informativa sobre el códec.

In [0]:
wavs = [to_wav16k(f) for f in audio_files()]   # el mismo audio en WAV 16 kHz mono
df_wav, m_wav = run_eval(asr, wavs, f"{BACKEND}_wav", fmt="wav")
for name, m in [("OPUS", m_opus), ("WAV", m_wav)]:
    print(f"{name}: WER={m.get('wer_corpus')}  RTF={m['rtf']:.3f}")

## C. Azure STT
Pon la salida de Azure del mismo conjunto en `AZURE_DIR/<nombre>.txt`. Tiempo y coste de Azure los aportas tú.

In [0]:
import mlflow
AZURE_RTF = None   # rellenar si se midió (segundos de proceso / segundos de audio)

refs, hyps = [], []
for f in audio_files():
    ref = read_ref(f)
    p = f"{AZURE_DIR}/{os.path.splitext(os.path.basename(f))[0]}.txt"
    if ref and os.path.exists(p):
        refs.append(NORM(ref)); hyps.append(NORM(open(p, encoding="utf-8").read()))

mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name="azure_stt_baseline"):
    mlflow.log_params({"model": "azure-stt", "audio_format": "opus", "n_audios": len(refs)})
    mlflow.log_metric("est_cost_eur_per_audio_hour", AZURE_EUR_PER_HOUR)
    if refs:
        mlflow.log_metric("wer_corpus", jiwer.wer(refs, hyps))
    if AZURE_RTF:
        mlflow.log_metric("rtf", AZURE_RTF)
print("Azure WER sobre", len(refs), "audios con referencia y salida de Azure")

Compara los runs en **Experiments → Compare**. Siguiente: `03_empaquetado_y_registro`.